# Module 19 - Agent loops

Use this notebook after `tests/test_agent.py` is passing. It starts with deterministic fake-backend checks for parsing, planning, scratchpad rendering, loop control, error recovery, and duplicate-action stops. The final section loads ProdLM for live ReAct runs against a real instruction model.

The deliverable is an agent failure-mode catalog: a few concrete transcripts, what went wrong, and what you would change in the prompt, tools, loop, or stop conditions.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys

from IPython.display import Markdown, display

from g2c.agent import (
    Action,
    Agent,
    AgentStep,
    NativeAgent,
    Observation,
    Plan,
    Scratchpad,
    extract_plan,
    parse_react_step,
    render_plan_block,
    render_planning_prompt,
    render_system_prompt,
)
from g2c.inference import (
    Backend,
    BackendInfo,
    InferenceResult,
    is_thinking_model,
    load_selected_backend,
)
from g2c.notebook_extras.sampling import printable
from g2c.tools import (
    Tool,
    ToolRegistry,
    make_calculator,
    make_read_file,
    make_run_python,
    run_with_tools,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

/Users/colkitt/sith/toys/courses/g2c


Run the agent tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 19 TODOs in `g2c/agent/`.

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_agent.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 19 agent tests are not passing yet."

........................................................................ [ 50%]
........................................................................ [100%]



## Display helpers

In [3]:
def short(text: Any, limit: int = 180) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def fake_inference(completion: str, *, prompt: str = "prompt") -> InferenceResult:
    info = BackendInfo(name="fake", model_id="fake-agent")
    return InferenceResult(
        prompt=prompt,
        completion=completion,
        prompt_tokens=len(prompt.split()),
        completion_tokens=len(completion.split()),
        latency_ms=1.0,
        backend=info,
    )


def show_parsed_step(label: str, text: str) -> None:
    parsed = parse_react_step(text)
    rows = [
        {
            "field": "thought",
            "value": parsed.thought,
        },
        {
            "field": "action",
            "value": "" if parsed.action is None else parsed.action.tool,
        },
        {
            "field": "arguments",
            "value": "" if parsed.action is None else json.dumps(parsed.action.arguments),
        },
        {
            "field": "final_answer",
            "value": "" if parsed.final_answer is None else parsed.final_answer,
        },
        {
            "field": "parse_error",
            "value": "" if parsed.parse_error is None else parsed.parse_error,
        },
    ]
    display(Markdown(f"### {label}\n\n" + markdown_table(rows, ["field", "value"])))


def show_plan(plan: Plan | None) -> None:
    if plan is None:
        display(Markdown("No plan parsed."))
        return
    rows = [{"part": "goal", "text": plan.goal}]
    for index, step in enumerate(plan.steps, start=1):
        rows.append({"part": str(index), "text": step})
    display(Markdown(markdown_table(rows, ["part", "text"])))


def show_agent_run(result, *, backend: Backend | None = None, show_prompts: bool = False) -> None:
    print("stopped:", result.stopped_reason)
    print("final answer:", printable(result.final_answer or "(none)"))
    if result.plan is not None:
        print("plan goal:", result.plan.goal)
    print()

    rows = []
    for index, step in enumerate(result.steps, start=1):
        rows.append(
            {
                "step": index,
                "thought": short(step.thought, 120),
                "action": "" if step.action is None else step.action.tool,
                "arguments": "" if step.action is None else json.dumps(step.action.arguments),
                "observation": "" if step.observation is None else short(step.observation.output, 160),
                "error": "" if step.observation is None else ("yes" if step.observation.is_error else "no"),
                "final": "" if step.final_answer is None else short(step.final_answer, 120),
                "parse_error": "" if step.parse_error is None else short(step.parse_error, 120),
            }
        )
    display(Markdown(markdown_table(rows, ["step", "thought", "action", "arguments", "observation", "error", "final", "parse_error"])))

    if backend is not None and hasattr(backend, "calls"):
        call_rows = []
        for index, call in enumerate(backend.calls, start=1):
            call_rows.append(
                {
                    "call": index,
                    "prompt chars": len(call["prompt"]),
                    "max_new_tokens": call["max_new_tokens"],
                    "temperature": call["temperature"],
                }
            )
        display(Markdown(markdown_table(call_rows, ["call", "prompt chars", "max_new_tokens", "temperature"])))
        if show_prompts:
            for index, call in enumerate(backend.calls, start=1):
                print(f"\n--- prompt {index} ---")
                print(short(call["prompt"], limit=1600))

## Build a local tool registry

The fake-backend exercises use the same registry a live ProdLM run will use: calculator, read-file, and Python execution. The read-file tool points at a small local directory created by this notebook.

In [4]:
scratch_dir = repo_root / "data" / "work" / "module19"
scratch_dir.mkdir(parents=True, exist_ok=True)

(scratch_dir / "numbers.txt").write_text("3\n7\n10\n20\n", encoding="utf-8")
(scratch_dir / "sales.csv").write_text(
    "item,units,price\nnotebook,3,12\npen,10,2\nbag,1,35\n",
    encoding="utf-8",
)
(scratch_dir / "long.txt").write_text("alpha " * 250, encoding="utf-8")

registry = ToolRegistry([
    make_calculator(),
    make_read_file(root=scratch_dir),
    make_run_python(timeout=3.0, cwd=scratch_dir),
])

print("registered:", registry.names())
print("scratch dir:", scratch_dir)

registered: ['calculator', 'read_file', 'run_python']
scratch dir: /Users/colkitt/sith/toys/courses/g2c/data/work/module19


In [5]:
print(render_system_prompt(registry.tools)[:1600])

You are a careful agent that solves problems step by step using tools.

You have access to the following tools:
  - calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.
    parameters schema:
      {
        "type": "object",
        "properties": {
          "expression": {
            "type": "string",
            "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."
          }
        },
        "required": [
          "expression"
        ]
      }
    use exactly: Action: calculator
  - read_file: Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory. Output is truncated server-side; the model does not control the length.
    parameters schema:
      {
        "type": "object",
        "properties": {
          "path": {
            "type": "string",
            "description": "Relative path under 

## Exercise 1 - Parse ReAct steps

Start with the wire format. A step is either `Thought` + `Action` + `Action Input`, or `Thought` + `Final Answer`. The parser should tolerate small formatting wobble but still reject malformed action inputs.

In [6]:
action_text = """Thought: I should use exact arithmetic.
Action: calculator
Action Input: {"expression": "23 * 17"}"""
final_text = """Thought: I now know the final answer.
Final Answer: 391"""
bad_text = """Thought: I should calculate.
Action: calculator
Action Input: {not json}"""
both_text = """Thought: I have enough information.
Action: calculator
Action Input: {"expression": "2 + 2"}
Final Answer: 4"""

show_parsed_step("Action step", action_text)
show_parsed_step("Final answer step", final_text)
show_parsed_step("Bad action input", bad_text)
show_parsed_step("Final answer wins", both_text)

### Action step

| field | value |
| --- | --- |
| thought | I should use exact arithmetic. |
| action | calculator |
| arguments | {"expression": "23 * 17"} |
| final_answer |  |
| parse_error |  |

### Final answer step

| field | value |
| --- | --- |
| thought | I now know the final answer. |
| action |  |
| arguments |  |
| final_answer | 391 |
| parse_error |  |

### Bad action input

| field | value |
| --- | --- |
| thought | I should calculate. |
| action |  |
| arguments |  |
| final_answer |  |
| parse_error | Action Input is not valid JSON: Expecting property name enclosed in double quotes |

### Final answer wins

| field | value |
| --- | --- |
| thought | I have enough information. |
| action |  |
| arguments |  |
| final_answer | 4 |
| parse_error |  |

## Exercise 2 - Extract and render a plan

The plan is a soft prior. It gets rendered into the prompt, but the model is not forced to follow it if observations reveal a better path.

In [7]:
plan_text = """Goal: Compute the average in numbers.txt.
1. Read numbers.txt.
2. Add the numbers and divide by the count.
3. Report the mean with the supporting arithmetic."""
plan = extract_plan(plan_text, "Read numbers.txt and compute the average.")
show_plan(plan)

if plan is not None:
    print(render_plan_block(plan.goal, plan.steps))

print("\nPlanning prompt excerpt:\n")
print(render_planning_prompt("Read numbers.txt and compute the average.", registry.tools)[:1200])

| part | text |
| --- | --- |
| goal | Compute the average in numbers.txt. |
| 1 | Read numbers.txt. |
| 2 | Add the numbers and divide by the count. |
| 3 | Report the mean with the supporting arithmetic. |

Plan:
Goal: Compute the average in numbers.txt.
1. Read numbers.txt.
2. Add the numbers and divide by the count.
3. Report the mean with the supporting arithmetic.

Planning prompt excerpt:

You are a planning assistant. Given a user task and a list of available
tools, produce a short numbered plan of how you would solve it.

Tools available:
  - calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.
    parameters schema:
      {
        "type": "object",
        "properties": {
          "expression": {
            "type": "string",
            "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."
          }
        },
        "required": [
          "expression"
        ]
      }
    use exactly: Action: calculator
  - read_file: Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory. Output is truncate

## Exercise 3 - Render the scratchpad

The scratchpad is the agent's working memory. Each prior thought, action, input, and observation is rendered back into the next prompt.

In [8]:
sp = Scratchpad()
step = AgentStep(
    completion=action_text,
    thought="I should use exact arithmetic.",
    action=Action(tool="calculator", arguments={"expression": "23 * 17"}),
    observation=Observation(output="391", is_error=False),
    final_answer=None,
    parse_error=None,
    inference=fake_inference(action_text),
)
sp.append(step)
print(sp.render())

Thought: I should use exact arithmetic.
Action: calculator
Action Input: {"expression": "23 * 17"}
Observation: 391


In [9]:
error_completion = """Thought: I used the wrong argument.
Action: calculator
Action Input: {"expr": "23 * 17"}"""
error_step = AgentStep(
    completion=error_completion,
    thought="I used the wrong argument.",
    action=Action(tool="calculator", arguments={"expr": "23 * 17"}),
    observation=Observation(output="unknown arguments: ['expr']", is_error=True),
    final_answer=None,
    parse_error=None,
    inference=fake_inference(error_completion),
)
sp.append(error_step)
print(sp.render())

Thought: I should use exact arithmetic.
Action: calculator
Action Input: {"expression": "23 * 17"}
Observation: 391

Thought: I used the wrong argument.
Action: calculator
Action Input: {"expr": "23 * 17"}
Observation: [error] unknown arguments: ['expr']


In [10]:
small_sp = Scratchpad(max_chars=180)
small_sp.append(step)
small_sp.append(error_step)
print(small_sp.render())

Thought: I used the wrong argument.
Action: calculator
Action Input: {"expr": "23 * 17"}
Observation: [error] unknown arguments: ['expr']


## Exercise 4 - Run the agent with a fake backend

Before involving a real model, use deterministic completions. This isolates the loop contract: prompt, parse, dispatch, observe, render scratchpad, repeat.

In [11]:
class FakeBackend(Backend):
    def __init__(self, completions: list[str], *, model_id: str = "fake-agent") -> None:
        self._completions = list(completions)
        self._info = BackendInfo(name="fake", model_id=model_id)
        self.calls: list[dict[str, Any]] = []

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        if not self._completions:
            raise RuntimeError("FakeBackend has no completions left")
        completion = self._completions.pop(0)
        self.calls.append(
            {
                "prompt": prompt,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
                "top_k": top_k,
                "top_p": top_p,
            }
        )
        return InferenceResult(
            prompt=prompt,
            completion=completion,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(completion.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [12]:
fake_backend = FakeBackend(
    [
        """Thought: I should calculate exactly.
Action: calculator
Action Input: {"expression": "23 * 17"}""",
        """Thought: I now know the final answer.
Final Answer: 23 * 17 = 391.""",
    ]
)

fake_agent = Agent(fake_backend, registry, plan=False, max_steps=4, temperature=0.0)
fake_result = fake_agent.run("What is 23 times 17?")
show_agent_run(fake_result, backend=fake_backend)

print("Second prompt includes the first observation:")
print(short(fake_backend.calls[-1]["prompt"], limit=1200))

stopped: final_answer
final answer: 23 * 17 = 391.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | I should calculate exactly. | calculator | {"expression": "23 * 17"} | 391 | no |  |  |
| 2 | I now know the final answer. |  |  |  |  | 23 * 17 = 391. |  |

| call | prompt chars | max_new_tokens | temperature |
| --- | --- | --- | --- |
| 1 | 4068 | 512 | 0.0 |
| 2 | 4182 | 512 | 0.0 |

Second prompt includes the first observation:
You are a careful agent that solves problems step by step using tools.\n\nYou have access to the following tools:\n  - calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.\n    parameters schema:\n      {\n        "type": "object",\n        "properties": {\n          "expression": {\n            "type": "string",\n            "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."\n          }\n        },\n        "required": [\n          "expression"\n        ]\n      }\n    use exactly: Action: calculator\n  - read_file: Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory. Output is truncated server-side; the model does not control the length.\n    parameters schema:\n      {\n        "type": "object",\n        "properties": {\n          "path": {\n        

## Exercise 5 - Recover from a tool error

The first action uses the wrong argument name. The dispatcher turns validation failure into an error observation, and the next model turn can correct itself.

In [13]:
recovery_backend = FakeBackend(
    [
        """Thought: I should calculate exactly.
Action: calculator
Action Input: {"expr": "23 * 17"}""",
        """Thought: The observation says the argument name was wrong.
Action: calculator
Action Input: {"expression": "23 * 17"}""",
        """Thought: I now know the final answer.
Final Answer: 391.""",
    ],
    model_id="fake-recovery",
)

recovery_agent = Agent(recovery_backend, registry, plan=False, max_steps=5, temperature=0.0)
recovery_result = recovery_agent.run("What is 23 times 17?")
show_agent_run(recovery_result, backend=recovery_backend)

stopped: final_answer
final answer: 391.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | I should calculate exactly. | calculator | {"expr": "23 * 17"} | missing required arguments: ['expression'] | yes |  |  |
| 2 | The observation says the argument name was wrong. | calculator | {"expression": "23 * 17"} | 391 | no |  |  |
| 3 | I now know the final answer. |  |  |  |  | 391. |  |

| call | prompt chars | max_new_tokens | temperature |
| --- | --- | --- | --- |
| 1 | 4068 | 512 | 0.0 |
| 2 | 4223 | 512 | 0.0 |
| 3 | 4359 | 512 | 0.0 |

## Exercise 6 - Stress-test loop detection

Duplicate-action detection is a heuristic: same tool, same arguments, two steps in a row. It catches a common runaway pattern, but you can disable it for legitimate retry workflows.

In [14]:
duplicate_completion = """Thought: I should try the same calculation again.
Action: calculator
Action Input: {"expression": "2 + 2"}"""
looping_backend = FakeBackend([duplicate_completion, duplicate_completion, duplicate_completion], model_id="fake-loop")
looping_agent = Agent(looping_backend, registry, plan=False, max_steps=5, loop_detection=True)
looping_result = looping_agent.run("What is 2 + 2?")
show_agent_run(looping_result, backend=looping_backend)

stopped: duplicate_action
final answer: (none)



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | I should try the same calculation again. | calculator | {"expression": "2 + 2"} | 4 | no |  |  |
| 2 | I should try the same calculation again. | calculator | {"expression": "2 + 2"} | 4 | no |  |  |

| call | prompt chars | max_new_tokens | temperature |
| --- | --- | --- | --- |
| 1 | 4062 | 512 | 0.2 |
| 2 | 4185 | 512 | 0.2 |

In [15]:
retry_backend = FakeBackend(
    [
        duplicate_completion,
        duplicate_completion,
        """Thought: I now know the final answer.
Final Answer: 4.""",
    ],
    model_id="fake-retry",
)
retry_agent = Agent(retry_backend, registry, plan=False, max_steps=5, loop_detection=False)
retry_result = retry_agent.run("What is 2 + 2?")
show_agent_run(retry_result, backend=retry_backend)

stopped: final_answer
final answer: 4.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | I should try the same calculation again. | calculator | {"expression": "2 + 2"} | 4 | no |  |  |
| 2 | I should try the same calculation again. | calculator | {"expression": "2 + 2"} | 4 | no |  |  |
| 3 | I now know the final answer. |  |  |  |  | 4. |  |

| call | prompt chars | max_new_tokens | temperature |
| --- | --- | --- | --- |
| 1 | 4062 | 512 | 0.2 |
| 2 | 4185 | 512 | 0.2 |
| 3 | 4308 | 512 | 0.2 |

## Exercise 7 - Add a planning phase

With `plan=True`, the backend is called once before the main loop to produce a numbered plan. The plan is then visible in every ReAct prompt.

In [16]:
planned_backend = FakeBackend(
    [
        """Goal: Compute the average from numbers.txt.
1. Read numbers.txt.
2. Use arithmetic to compute the average.
3. Report the result.""",
        """Thought: I need the file contents first.
Action: read_file
Action Input: {"path": "numbers.txt"}""",
        """Thought: I should compute the mean from the observed numbers.
Action: calculator
Action Input: {"expression": "(3 + 7 + 10 + 20) / 4"}""",
        """Thought: I now know the final answer.
Final Answer: The average is 10.""",
    ],
    model_id="fake-planned",
)

planned_agent = Agent(planned_backend, registry, plan=True, max_steps=5, temperature=0.0)
planned_result = planned_agent.run("Read numbers.txt and tell me the average.")
show_plan(planned_result.plan)
show_agent_run(planned_result, backend=planned_backend)

print("First ReAct prompt after the planning call:")
print(short(planned_backend.calls[1]["prompt"], limit=1600))

| part | text |
| --- | --- |
| goal | Compute the average from numbers.txt. |
| 1 | Read numbers.txt. |
| 2 | Use arithmetic to compute the average. |
| 3 | Report the result. |

stopped: final_answer
final answer: The average is 10.
plan goal: Compute the average from numbers.txt.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | I need the file contents first. | read_file | {"path": "numbers.txt"} | 3\n7\n10\n20\n | no |  |  |
| 2 | I should compute the mean from the observed numbers. | calculator | {"expression": "(3 + 7 + 10 + 20) / 4"} | 10.0 | no |  |  |
| 3 | I now know the final answer. |  |  |  |  | The average is 10. |  |

| call | prompt chars | max_new_tokens | temperature |
| --- | --- | --- | --- |
| 1 | 2771 | 256 | 0.0 |
| 2 | 4225 | 512 | 0.0 |
| 3 | 4347 | 512 | 0.0 |
| 4 | 4501 | 512 | 0.0 |

First ReAct prompt after the planning call:
You are a careful agent that solves problems step by step using tools.\n\nYou have access to the following tools:\n  - calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.\n    parameters schema:\n      {\n        "type": "object",\n        "properties": {\n          "expression": {\n            "type": "string",\n            "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."\n          }\n        },\n        "required": [\n          "expression"\n        ]\n      }\n    use exactly: Action: calculator\n  - read_file: Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory. Output is truncated server-side; the model does not control the length.\n    parameters schema:\n      {\n        "type": "object",\n        "properties": {\n          "path": {\n          

## Exercise 8 - Scratchpad cap

A character cap drops old scratchpad blocks when the rendered history grows too long. This is crude, but it makes the context-management issue concrete.

In [17]:
cap_backend = FakeBackend(
    [
        """Thought: I should read the long file.
Action: read_file
Action Input: {"path": "long.txt", "max_chars": 1200}""",
        """Thought: I should do a tiny calculation.
Action: calculator
Action Input: {"expression": "10 + 5"}""",
        """Thought: I now know the final answer.
Final Answer: I read the long file and computed 15.""",
    ],
    model_id="fake-cap",
)

cap_agent = Agent(cap_backend, registry, plan=False, max_steps=4, scratchpad_max_chars=450)
cap_result = cap_agent.run("Read long.txt, then compute 10 + 5.")
show_agent_run(cap_result, backend=cap_backend)

for i, call in enumerate(cap_backend.calls, start=1):
    print(f"prompt {i}: {len(call['prompt'])} characters")

stopped: final_answer
final answer: I read the long file and computed 15.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | I should read the long file. | read_file | {"path": "long.txt", "max_chars": 1200} | unknown arguments: ['max_chars'] | yes |  |  |
| 2 | I should do a tiny calculation. | calculator | {"expression": "10 + 5"} | 15 | no |  |  |
| 3 | I now know the final answer. |  |  |  |  | I read the long file and computed 15. |  |

| call | prompt chars | max_new_tokens | temperature |
| --- | --- | --- | --- |
| 1 | 4083 | 512 | 0.2 |
| 2 | 4248 | 512 | 0.2 |
| 3 | 4364 | 512 | 0.2 |

prompt 1: 4083 characters
prompt 2: 4248 characters
prompt 3: 4364 characters


## Exercise 9 - Load a live backend for agent runs

The default live backend is ProdLM. To test a course-trained model instead, set `MODEL_SELECTION = "course"` for the strongest course artifact, or set it to a base artifact name like `"TinyLLM-30M"`; the loader will prefer `-DPO`, then `-SFT`, then the base artifact.

In [18]:
MODEL_SELECTION = "ProdLM"  # "ProdLM", "course", or an artifact base/name such as "TinyLLM-30M"
PRODLM_MODEL_ID = None  # optional Ollama tag override when MODEL_SELECTION == "ProdLM"
LIVE_DEVICE = "auto"
LIVE_TORCH_DTYPE = "float16"

live_backend = None
try:
    live_backend = load_selected_backend(
        MODEL_SELECTION,
        repo_root=repo_root,
        prodlm_model_id=PRODLM_MODEL_ID,
        device=LIVE_DEVICE,
        torch_dtype=LIVE_TORCH_DTYPE,
        required=False,
    )
    if live_backend is None:
        print("No live backend loaded. Run ./prodlm.sh or choose an available artifact.")
    else:
        print("loaded:", live_backend.info)
except Exception as exc:
    print(f"Live backend unavailable: {type(exc).__name__}: {exc}")

loaded: BackendInfo(name='prodlm', model_id='llama3.2:3b', extra={'base_url': 'http://localhost:11434', 'configured_name': 'ProdLM'})


In [19]:
def run_live_agent(
    question: str,
    *,
    tools: ToolRegistry = registry,
    plan: bool = True,
    max_steps: int = 6,
    max_new_tokens: int = 384,
    temperature: float = 0.0,
    scratchpad_max_chars: int | None = None,
    loop_detection: bool = True,
):
    """ReAct agent — text-format, the lesson of this module."""
    if live_backend is None:
        print("Skipping live run: No live backend is loaded.")
        return None
    agent = Agent(
        live_backend,
        tools,
        plan=plan,
        max_steps=max_steps,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        scratchpad_max_chars=scratchpad_max_chars,
        loop_detection=loop_detection,
    )
    try:
        result = agent.run(question)
    except Exception as exc:  # Live local servers can be offline or mid-pull.
        print(type(exc).__name__ + ":", exc)
        return None
    show_agent_run(result)
    return result


def run_live_native_agent(
    question: str,
    *,
    tools: ToolRegistry = registry,
    plan: bool = False,
    max_steps: int = 8,
    max_new_tokens: int = 512,
    temperature: float = 0.0,
    loop_detection: bool = True,
):
    """Structured-tool-calling agent — same loop, native wire format.

    Speaks Ollama's `/api/chat` + `tool_calls` protocol instead of ReAct
    text markers. The model emits structured JSON tool calls directly,
    so there is no ReAct parser in the path. Use this to compare how
    the same model handles the same task on each channel.
    """
    if live_backend is None:
        print("Skipping live run: No live backend is loaded.")
        return None
    think_setting = False if is_thinking_model(live_backend.info.model_id) else None
    agent = NativeAgent(
        live_backend,
        tools,
        plan=plan,
        max_steps=max_steps,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        loop_detection=loop_detection,
        think=think_setting,
    )
    try:
        result = agent.run(question)
    except Exception as exc:
        print(type(exc).__name__ + ":", exc)
        return None
    show_agent_run(result)
    return result

## Exercise 10 - Live one-shot calculator

This should be an easy case. The task needs one tool call, so `plan=False` is usually cleaner.

In [ ]:
live_calc_question = "Use the calculator to compute (1847 * 29) - 138. Then give the final integer."
live_calc_result = run_live_native_agent(
    live_calc_question,
    tools=ToolRegistry([make_calculator()]),
    plan=False,
    max_steps=4,
)

## Exercise 11 - Live multi-step file task

Now give the model a task where it must read local data before answering. Try `plan=False` and `plan=True`; the difference is often visible in how direct the tool sequence is.

In [23]:
file_question = "Read numbers.txt and report both the sum and the mean. Use tools for the file read and arithmetic."
file_result_without_plan = run_live_native_agent(file_question, plan=False, max_steps=6)
file_result_with_plan = run_live_native_agent(file_question, plan=True, max_steps=6)

stopped: final_answer
final answer: The sum of the numbers in the file is 10.0 and the mean is 40.0.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 |  | run_python | {"code": "import math; print(math.fsum([float(line) for line in open(\"numbers.txt\").read().split()]))\nprint(math.fmean([float(line) for line in open(\"numbers.txt\").read().split()]))"} | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 2, in <module>\nAttributeError: module 'math' has no attribute 'fmean' | no |  |  |
| 2 | It seems that the `fmean` function does not exist in the math module. Instead, you can use the `fmean` function from ... | run_python | {"code": "import statistics; print(statistics.fmean([float(line) for line in open(\"numbers.txt\").read().split()]))\nprint(statistics.fsum([float(line) for line in open(\"numbers.txt\").read().split()]))"} | 10.0\n40.0\n | no |  |  |
| 3 |  |  |  |  |  | The sum of the numbers in the file is 10.0 and the mean is 40.0. |  |

stopped: final_answer
final answer: It seems like the calculator tool is having trouble with the expression. Let me try to break it down further.

The sum of the numbers in the file is 40.
The mean of the numbers in the file is 8.0.
plan goal: Read numbers.txt, calculate sum and mean of its contents, and report both values.



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 |  | read_file | {"path": "numbers.txt"} | 3\n7\n10\n20\n | no |  |  |
| 2 | The sum of the numbers in numbers.txt is 40.\n\nTo calculate the mean, we can use the calculator tool:\n\n{"name": "c... | calculator | {"expression": "sum / len(lines)"} | AST node not allowed: Name | yes |  |  |
| 3 |  |  |  |  |  | It seems like the calculator tool is having trouble with the expression. Let me try to break it down further.\n\nThe ... |  |

## Exercise 12 - Compare Module 18 and Module 19

Module 18's loop waits for `<tool_call>` blocks. Module 19 asks the model for `Thought` / `Action` / `Action Input` turns. Run both on the same task and compare reliability, prompt length, and final answer quality.

In [29]:
comparison_question = "Read sales.csv and compute total revenue by combining units and price."
comparison_registry = ToolRegistry([
    #make_calculator(),
    make_read_file(root=scratch_dir),
    make_run_python(timeout=3.0, cwd=scratch_dir),
])

if live_backend is None:
    print("Skipping comparison: No live backend is loaded.")
else:
    try:
        tool_loop_result = run_with_tools(
            live_backend,
            comparison_registry,
            comparison_question,
            max_steps=5,
            max_new_tokens=384,
            temperature=0.0,
        )
        print("Module 18 stopped:", tool_loop_result.stopped_reason)
        print(printable(tool_loop_result.final_answer or "(none)"))
    except Exception as exc:
        print("Module 18 loop failed:", type(exc).__name__ + ":", exc)

    print("\nModule 19 agent:")
    agent_result = run_live_native_agent(comparison_question, tools=comparison_registry, plan=True, max_steps=6)

Module 18 stopped: no_more_calls
It seems that the tool was unable to find a file named "data" in the current working directory. The code is trying to read from this file, but since it doesn't exist, it's throwing an error.

To fix this, you can either create a file named "data.csv" (or whatever your actual filename is) and write the data into it, or modify the code to use a different method of reading the CSV file. Here's how you could do that:

```python
import pandas as pd

# Read the csv file directly from the input stream
df = pd.read_csv('sales.csv')

# Compute total revenue by combining units and price
total_revenue = df['Units'] * df['Price'].sum()

print(total_revenue)
```

This code will read the 'sales.csv' file directly from the input stream, compute the total revenue, and print it out.

Module 19 agent:
stopped: final_answer
final answer: It seems that the tool output is an error message instead of the result. Let me try again.

To compute total revenue by combining units 

| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 |  | run_python | {"code": "import pandas as pd; print(pd.read_csv(data).df['total_revenue'] = pd.read_csv(data).df['units']*pd.read_csv(data).df['price']); print(pd.read_csv(data).df['total_revenue'])"} | [run_python: exit 1]\nFile "<string>", line 1\n    import pandas as pd; print(pd.read_csv(data).df['total_revenue'] = pd.read_csv(data).df['units']*pd.read_c... | no |  |  |
| 2 |  |  |  |  |  | It seems that the tool output is an error message instead of the result. Let me try again.\n\nTo compute total revenu... |  |

## Exercise 12b - Same agent, structured tool-calling channel

The `NativeAgent` runs the same observe → act → observe loop as `Agent`, but speaks the modern structured tool-calling protocol on the wire. Each turn, instead of regex-parsing `Thought:` / `Action:` markers out of plain text, the harness wraps the step history into an OpenAI/Ollama-style `messages` list with `tool_calls` and `role: "tool"` entries, and unwraps the model's structured response back into an `AgentStep`. Same `Action` / `Observation` / `AgentStep` data shapes, same loop control, same planner — only the I/O at the model boundary differs.

Production agent frameworks (Cursor, Claude Code, OpenAI assistants, Anthropic SDK) all use this format. ReAct still appears in research papers and is the buildable lesson for this module, but the structured channel is what models are actually trained for in 2025–2026: provider-specific delimiters (`<|python_tag|>` for Llama, `<tool_call>` for Qwen, `tool_use` for Anthropic) get parsed out of the model's text by the runtime and surfaced as structured `tool_calls`.

Run the same task through both channels and compare. Expect the native channel to fail less often on small models — the format is in the post-training distribution, the ReAct format mostly isn't.

In [26]:
native_question = "Read sales.csv and compute total revenue."
native_registry = ToolRegistry([
    make_calculator(),
    make_read_file(root=scratch_dir),
    make_run_python(timeout=3.0, cwd=scratch_dir),
])

print("ReAct agent (Module 19 default):")
react_result = run_live_agent(native_question, tools=native_registry, plan=False, max_steps=8)

print("\nNative agent (structured tool calling):")
native_result = run_live_native_agent(native_question, tools=native_registry, plan=False, max_steps=8)

ReAct agent (Module 19 default):
stopped: duplicate_action
final answer: (none)



| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 |  | read_file | {"path": "sales.csv"} | item,units,price\nnotebook,3,12\npen,10,2\nbag,1,35\n | no |  |  |
| 2 | I should use the run_python action to execute a Python code that reads the sales.csv file and computes the total reve... | run_python | {"code": "import pandas as pd; print(pd.read_csv(data).df['units'].sum() * pd.read_csv(data).df['price'].mean())", "path": "sales.csv"} | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 2, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/pyth... | no |  |  |
| 3 | I should use the run_python action to execute a Python code that reads the sales.csv file and computes the total reve... | run_python | {"code": "import pandas as pd; print(pd.read_csv(data).df['units'].sum() * pd.read_csv(data).df['price'].mean())", "path": "sales.csv"} | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 2, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/pyth... | no |  |  |


Native agent (structured tool calling):
stopped: final_answer
final answer: The 'Revenue' column does not exist in the sales.csv file. The code is trying to access a column that does not exist.

To find the total revenue, you can try to identify the correct column name that represents revenue in the csv file. 

Alternatively, if the csv file has multiple columns representing different types of revenue (e.g., 'Revenue1', 'Revenue2', etc.), you would need to specify which one you want to use.

Here is an example of how you could modify the code to find the correct column name:

```
import pandas as pd

# Read the csv file
data = pd.read_csv('sales.csv')

# Get a list of all column names
column_names = data.columns.tolist()

# Print the column names
print(column_names)

# Ask the user to input the correct column name
correct_column_name = input("Please enter the correct column name: ")

# Calculate the total revenue
total_revenue = data[correct_column_name].sum()

# Print the result
prin

| step | thought | action | arguments | observation | error | final | parse_error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 |  | run_python | {"code": "import pandas as pd; print(pd.read_csv(data).loc[:, \"Revenue\"].sum())"} | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\nNameError: name 'data' is not defined | no |  |  |
| 2 | It seems like the tool output was incomplete. Let me try again.\n\nHere's another attempt at calling the Python tool ... | run_python | {"code": "import pandas as pd; data = pd.read_csv(\"sales.csv\"); print(data.loc[:, \"Revenue\"].sum())"} | [run_python: exit 1]\nTraceback (most recent call last):\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages/pandas/core/indexes... | no |  |  |
| 3 |  |  |  |  |  | The 'Revenue' column does not exist in the sales.csv file. The code is trying to access a column that does not exist.... |  |

## Exercise 13 - Build a failure-mode catalog

Run this optional cell with your local model after the earlier live cells work. Keep a short catalog of what happened: parse failures, wrong tool choice, repeated actions, premature final answers, or good recoveries from tool errors.

Requires a live backend so that `run_live_native_agent` can call the model. Skip the cell if one is not loaded.

In [ ]:
failure_questions = [
    "Read numbers.txt and compute the largest number minus the smallest number.",
    "Read sales.csv and compute total revenue. Then explain the arithmetic.",
    "Use the calculator to compute 19 ** 3 + 7 ** 2.",
    "Try to read missing.txt. If it fails, explain what failed instead of inventing contents.",
]

failure_runs = []
for question in failure_questions:
    print("=" * 72)
    print(question)
    result = run_live_native_agent(question, tools=registry, plan=True, max_steps=6, temperature=0.0)
    failure_runs.append((question, result))

## Postmortem notes

Use this structure for your deliverable:

1. **Best success case:** paste the task, final answer, and the step table. Explain why it worked.
2. **Most informative failure:** paste the task and the step table. Categorize the failure: bad parse, wrong tool, invalid arguments, duplicate action, missing context, premature final answer, or model refusal.
3. **One loop change:** propose a change to the prompt, parser, tools, scratchpad cap, planning toggle, or stop condition.
4. **Module 18 comparison:** for one task, say whether the plain tool loop or ReAct agent was easier for the model to use.
5. **Channel comparison (Exercise 12b):** for the same task, did the ReAct `Agent` or the structured-format `NativeAgent` produce a cleaner trajectory? Name the specific failure mode you saw on the worse channel — bad parse, wrong format, empty response, etc. — and tie it to what the model was trained for.

The goal is not to prove the agent is robust. The goal is to see exactly where the wrapper helps and where the model is still the bottleneck — including the bottleneck of "ReAct is a parser convention from 2022; the model has been post-trained for the structured format since 2023."